# Housing, economy and public services
## Mexicali Urban Liveability Index — `WP08_housing_economy_and_services`

**Lead:** TBC
**Indicators assigned:** 4
**Schema version:** 1.0.0

Housing affordability, land and house prices, jobs, and the quality/adequacy of public services, from census, ENIGH/ENVI, DENUE, cadastral and municipal sources.

Survey-based measures are often only representative at city or AGEB scale. Report at the true scale and flag replication downward rather than implying fine-grained variation.

> New to this project? Work through
> [`00_overview_and_schema.ipynb`](00_overview_and_schema.ipynb)
> first — it carries one indicator end to end. Then read
> [`docs/analyst_guide.md`](../docs/analyst_guide.md).

## Before you start

**Your job, in one sentence:** for each indicator below, produce a
number for every unit of geography, and the documentation that
makes that number defensible.

Work in this order. Steps 1 and 2 shape everything after
them, so they are worth doing before opening any data.

| | Step | Where to look |
|---|---|---|
| 1 | Write the causal pathway: *X changes a mechanism, which changes a behaviour or exposure, which affects health* | [guide §2.1](../docs/analyst_guide.md) |
| 2 | Find **independent health evidence** for that pathway, with an effect size | [guide §2.2](../docs/analyst_guide.md) |
| 3 | Find the data. Record citation, URL, date retrieved, licence | [guide §3](../docs/analyst_guide.md) |
| 4 | Compute it — pick the matching recipe | [cookbook](00b_cookbook.ipynb) |
| 5 | `harmonise` → `label` → `check` → `write_indicator` | [cookbook §9](00b_cookbook.ipynb) |
| 6 | Look at a map of your result before calling it done | [guide §6](../docs/analyst_guide.md) |

Three things worth knowing now:

- **The evidence matters as much as the calculation.** The
  article an indicator was adapted from shows it has been *used*,
  not that it affects health. Supplying that link is a substantive
  part of the work.
- **Deliver natural units.** No normalising, no reverse-coding —
  set `direction` instead. The index does that once, centrally.
- **Cover Condesa.** Compute on `grid_100m` where your data allow
  it: that reaches all 40 fraccionamientos of the development,
  where `manzana` reaches only 33. The validator treats poor
  Condesa coverage as an error.

New to this? Part 1 of [`00b_cookbook.ipynb`](00b_cookbook.ipynb)
works one indicator through the whole process, on data already in
the repository.

---
## Setup

In [ ]:
import os
import sys

sys.path.insert(0, os.path.abspath('..'))

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import uli

# Identify yourself once; it is copied into every deliverable.
ANALYST = {
    'name': 'TODO: your name',
    'email': None,
    'institution': None,
}

print(f'ULI schema version {uli.SCHEMA_VERSION}')
print(f'Reference geographies available: {uli.geography.available()}')

WORK_PACKAGE = 'WP08_housing_economy_and_services'
NOTEBOOK = 'notebooks/08_housing_economy_and_services.ipynb'

---
## Your indicators (4)

Each has a brief reproducing what the workbook records,
then three working cells: documentation, calculation,
delivery.

### 289 — Housing affordability

`housing_affordability` · *Housing · Affordable Housing · Housing affordability*

- **Lenses to deliver:** accessibility, quantity
- **Draft rationale (rewrite this):** Affordable housing is essential for liveability and health as high housing costs limit resources for other social determinants of health like food and healthcare.
- **Adapted from:** #22: Alderton (2019), ‘What is the meaning of urban liveability for a city in a low-to-middle-income country? Contextualising liveability for Bangkok, Thailand’; #27: Alzaim (2024), ‘Integrated Framework for Enhancing Liveability and Ecological Sustainability in UAE Communities’
- **Candidate data sources:** "con la ENIGH se puede calcular gasto de alquiler entre ingresos" https://www.inegi.org.mx/programas/enigh/nc/2024/ "Encuesta Nacional de Vivienda (ENVI) 2020" https://www.inegi.org.mx/programas/envi/2020/ " Tipología de vivienda (2024)" from the Actualizaciones del territorio maps https://www.mexicali.gob.mx/sitioimip/geovisor
- **Team notes:** ER: Data will only be available at the city-level from the "Encuesta Nacional de Vivienda (ENVI) 2020" https://www.inegi.org.mx/programas/envi/2020/ We could also use the categories from the " Tipología de vivienda (2024)" from the Actualizaciones del territorio maps https://www.mexicali.gob.mx/sitioimip/geovisor

> **Your first task is not GIS.** Write the causal pathway sentence, then find independent health evidence for it — see the framing section above.

In [ ]:
# 1. Documentation --------------------------------------------
# Pre-filled from the workbook; edit in place.  Run
# uli.todos(meta_289) at any time to list what is
# still outstanding.
meta_289 = uli.metadata_stub(289, analyst=ANALYST)

# meta_289['rationale']['statement'] = """..."""
# meta_289['rationale']['health_pathways'] = [
#     'physical_activity_transport',
# ]
# meta_289['rationale']['arid_context'] = '...'
# meta_289['rationale']['evidence'] = [
#     {
#         'claim': '...',
#         'citation': '...',
#         'doi': '...',
#         'evidence_type': 'systematic_review',
#         'population': '...',
#         'exposure': '...',
#         'outcome': '...',
#         'effect': 'RR 0.92 (95% CI 0.88-0.96) per ...',
#         'threshold_support': '...',
#     },
# ]
# meta_289['data_sources'] = [
#     {
#         'name': '...',
#         'custodian': '...',
#         'citation': '...',
#         'url': '...',
#         'date_retrieved': 'YYYY-MM-DD',
#         'licence': '...',
#         'redistributable': True,
#         'spatial_resolution': '...',
#         'temporal_coverage': '...',
#         'condesa_coverage': 'full',
#     },
# ]

uli.todos(meta_289)

In [ ]:
# 2. Calculation ----------------------------------------------
# Produce a DataFrame with two columns: geo_id and value.
#
# Suggested starting point: cookbook Recipe 7
# (00b_cookbook.ipynb).  If it does not fit, the other recipes
# are in the same notebook -- this is a guess from the
# indicator's lens and category, not a decision.
#
# grid_100m reaches all 40 Condesa fraccionamientos;
# manzana reaches only 33.
NATIVE_SCALE = 'ageb'
METHOD = 'population_weighted_mean'

# Uncomment and adapt:
# features = gpd.read_file('../data/raw/your_data.gpkg')
# Values probably already exist per AGEB or manzana:
# table = pd.read_csv('../data/raw/your_file.csv',
#                     dtype={'CVEGEO': str})
# native = table.rename(columns={'CVEGEO': 'geo_id',
#                                'your_column': 'value'}
#                       )[['geo_id', 'value']]

native_289 = None  # <- assign your result here

In [ ]:
# 3. Harmonise, validate, deliver ------------------------------
harmonised_289 = uli.harmonise(
    native_289,
    native_scale=NATIVE_SCALE,
    method=METHOD,
)
results_289 = uli.label(
    harmonised_289,
    meta_289,
    measure_id='housing_affordability__accessibility',
)
# Several measures?  Label each and combine:
#   results = uli.assemble([results_a, results_b])

print(uli.check(results_289, meta_289))
# uli.write_indicator(results_289, meta_289)

### 290 — Land and house price

`land_and_house_price` · *Housing · Affordable Housing · Land and house price*

- **Lenses to deliver:** quantity
- **Draft rationale (rewrite this):** High housing and land prices are linked to reduced affordability and financial stress, which can negatively impact residents' overall quality of life and residential stability.
- **Adapted from:** #16: Kashi (2025), ‘Spatial analysis and ranking of urban districts based on a comprehensive livability approach: the case of Tehran’
- **Candidate data sources:** "con la ENIGH se puede calcular gasto de alquiler entre ingresos" https://www.inegi.org.mx/programas/enigh/nc/2024/ "Encuesta Nacional de Vivienda (ENVI) 2020" https://www.inegi.org.mx/programas/envi/2020/ " Tipología de vivienda (2024)" from the Actualizaciones del territorio maps https://www.mexicali.gob.mx/sitioimip/geovisor
- **Team notes:** ER: Juan, el catastro tiene costo del lote y vivienda? https://tecmx.sharepoint.com/:f:/s/Mexicali-SIUM/IgCo34MzoO2pQ6n_8_fN7n7yAXbMN8SNG9AALvIBr0s7taI?e=R4YpGb https://tecmx.sharepoint.com/:f:/s/Mexicali-SIUM/IgCIU9bPtEF1RI3-GeohnPsyAV8qQj9UYPAZZ3rpy6S1RZE?e=mcIQXk
- **Open questions raised:** CH: To judge what is "high" and unaffordable may require some interpretation / guidance / use of a threshold (in Australia we us a 30:40 measure of housing affordability stress https://www.ahuri.edu.au/analysis/brief/understanding-3040-indicator-housing-affordability-stress

> **Your first task is not GIS.** Write the causal pathway sentence, then find independent health evidence for it — see the framing section above.

In [ ]:
# 1. Documentation --------------------------------------------
# Pre-filled from the workbook; edit in place.  Run
# uli.todos(meta_290) at any time to list what is
# still outstanding.
meta_290 = uli.metadata_stub(290, analyst=ANALYST)

# meta_290['rationale']['statement'] = """..."""
# meta_290['rationale']['health_pathways'] = [
#     'physical_activity_transport',
# ]
# meta_290['rationale']['arid_context'] = '...'
# meta_290['rationale']['evidence'] = [
#     {
#         'claim': '...',
#         'citation': '...',
#         'doi': '...',
#         'evidence_type': 'systematic_review',
#         'population': '...',
#         'exposure': '...',
#         'outcome': '...',
#         'effect': 'RR 0.92 (95% CI 0.88-0.96) per ...',
#         'threshold_support': '...',
#     },
# ]
# meta_290['data_sources'] = [
#     {
#         'name': '...',
#         'custodian': '...',
#         'citation': '...',
#         'url': '...',
#         'date_retrieved': 'YYYY-MM-DD',
#         'licence': '...',
#         'redistributable': True,
#         'spatial_resolution': '...',
#         'temporal_coverage': '...',
#         'condesa_coverage': 'full',
#     },
# ]

uli.todos(meta_290)

In [ ]:
# 2. Calculation ----------------------------------------------
# Produce a DataFrame with two columns: geo_id and value.
#
# Suggested starting point: cookbook Recipe 3
# (00b_cookbook.ipynb).  If it does not fit, the other recipes
# are in the same notebook -- this is a guess from the
# indicator's lens and category, not a decision.
#
# grid_100m reaches all 40 Condesa fraccionamientos;
# manzana reaches only 33.
NATIVE_SCALE = 'grid_100m'
METHOD = 'area_weighted_mean'

# Uncomment and adapt:
# features = gpd.read_file('../data/raw/your_data.gpkg')
# native = uli.areal_share(polygons, NATIVE_SCALE, as_percentage=True)
# or, for a continuous surface:
# native = uli.zonal_statistic('raster.tif', NATIVE_SCALE, 'mean')

native_290 = None  # <- assign your result here

In [ ]:
# 3. Harmonise, validate, deliver ------------------------------
harmonised_290 = uli.harmonise(
    native_290,
    native_scale=NATIVE_SCALE,
    method=METHOD,
)
results_290 = uli.label(
    harmonised_290,
    meta_290,
    measure_id='land_and_house_price__quantity',
)
# Several measures?  Label each and combine:
#   results = uli.assemble([results_a, results_b])

print(uli.check(results_290, meta_290))
# uli.write_indicator(results_290, meta_290)

### 1 — Public services

`public_services` · *Social Infrastructure · Public Services · Community Facilities · Public services*

- **Lenses to deliver:** accessibility, quality
- **Draft rationale (rewrite this):** Access to telecommunication infrastructure and mobile network coverage is identified as a vital public service and a major interest for resident liveability.
- **Adapted from:** #8: Onnom (2018), ‘Development of a Liveable City Index (LCI) Using Multi Criteria Geospatial Modelling for Medium Class Cities in Developing Countries’
- **Open questions raised:** CH: is there much variation in access to telecommunications infrastructure in mexicali?

> **Your first task is not GIS.** Write the causal pathway sentence, then find independent health evidence for it — see the framing section above.

In [ ]:
# 1. Documentation --------------------------------------------
# Pre-filled from the workbook; edit in place.  Run
# uli.todos(meta_1) at any time to list what is
# still outstanding.
meta_1 = uli.metadata_stub(1, analyst=ANALYST)

# meta_1['rationale']['statement'] = """..."""
# meta_1['rationale']['health_pathways'] = [
#     'physical_activity_transport',
# ]
# meta_1['rationale']['arid_context'] = '...'
# meta_1['rationale']['evidence'] = [
#     {
#         'claim': '...',
#         'citation': '...',
#         'doi': '...',
#         'evidence_type': 'systematic_review',
#         'population': '...',
#         'exposure': '...',
#         'outcome': '...',
#         'effect': 'RR 0.92 (95% CI 0.88-0.96) per ...',
#         'threshold_support': '...',
#     },
# ]
# meta_1['data_sources'] = [
#     {
#         'name': '...',
#         'custodian': '...',
#         'citation': '...',
#         'url': '...',
#         'date_retrieved': 'YYYY-MM-DD',
#         'licence': '...',
#         'redistributable': True,
#         'spatial_resolution': '...',
#         'temporal_coverage': '...',
#         'condesa_coverage': 'full',
#     },
# ]

uli.todos(meta_1)

In [ ]:
# 2. Calculation ----------------------------------------------
# Produce a DataFrame with two columns: geo_id and value.
#
# Suggested starting point: cookbook Recipe 2
# (00b_cookbook.ipynb).  If it does not fit, the other recipes
# are in the same notebook -- this is a guess from the
# indicator's lens and category, not a decision.
#
# grid_100m reaches all 40 Condesa fraccionamientos;
# manzana reaches only 33.
NATIVE_SCALE = 'grid_100m'
METHOD = 'population_weighted_mean'

# Uncomment and adapt:
# features = gpd.read_file('../data/raw/your_data.gpkg')
# distance = uli.distance_to_nearest(features, NATIVE_SCALE)
# native = distance.assign(
#     value=(distance['value'] <= 500).astype(float) * 100)

native_1 = None  # <- assign your result here

In [ ]:
# 3. Harmonise, validate, deliver ------------------------------
harmonised_1 = uli.harmonise(
    native_1,
    native_scale=NATIVE_SCALE,
    method=METHOD,
)
results_1 = uli.label(
    harmonised_1,
    meta_1,
    measure_id='public_services__accessibility',
)
# Several measures?  Label each and combine:
#   results = uli.assemble([results_a, results_b])

print(uli.check(results_1, meta_1))
# uli.write_indicator(results_1, meta_1)

### 112 — Jobs

`jobs` · *Economic Development · Employment · Jobs*

- **Lenses to deliver:** quantity
- **Draft rationale (rewrite this):** Local employment opportunities allow residents to live and work within the same district, improving work-life balance and reducing the health burdens of long commutes.
- **Adapted from:** #22: Alderton (2019), ‘What is the meaning of urban liveability for a city in a low-to-middle-income country? Contextualising liveability for Bangkok, Thailand’
- **Team notes:** Pues sí uno tiene El valor de fuerza laboral sería mejor con ese y El cociente sobre los que pueden trabajar

> **Your first task is not GIS.** Write the causal pathway sentence, then find independent health evidence for it — see the framing section above.

In [ ]:
# 1. Documentation --------------------------------------------
# Pre-filled from the workbook; edit in place.  Run
# uli.todos(meta_112) at any time to list what is
# still outstanding.
meta_112 = uli.metadata_stub(112, analyst=ANALYST)

# meta_112['rationale']['statement'] = """..."""
# meta_112['rationale']['health_pathways'] = [
#     'physical_activity_transport',
# ]
# meta_112['rationale']['arid_context'] = '...'
# meta_112['rationale']['evidence'] = [
#     {
#         'claim': '...',
#         'citation': '...',
#         'doi': '...',
#         'evidence_type': 'systematic_review',
#         'population': '...',
#         'exposure': '...',
#         'outcome': '...',
#         'effect': 'RR 0.92 (95% CI 0.88-0.96) per ...',
#         'threshold_support': '...',
#     },
# ]
# meta_112['data_sources'] = [
#     {
#         'name': '...',
#         'custodian': '...',
#         'citation': '...',
#         'url': '...',
#         'date_retrieved': 'YYYY-MM-DD',
#         'licence': '...',
#         'redistributable': True,
#         'spatial_resolution': '...',
#         'temporal_coverage': '...',
#         'condesa_coverage': 'full',
#     },
# ]

uli.todos(meta_112)

In [ ]:
# 2. Calculation ----------------------------------------------
# Produce a DataFrame with two columns: geo_id and value.
#
# Suggested starting point: cookbook Recipe 7
# (00b_cookbook.ipynb).  If it does not fit, the other recipes
# are in the same notebook -- this is a guess from the
# indicator's lens and category, not a decision.
#
# grid_100m reaches all 40 Condesa fraccionamientos;
# manzana reaches only 33.
NATIVE_SCALE = 'ageb'
METHOD = 'population_weighted_mean'

# Uncomment and adapt:
# features = gpd.read_file('../data/raw/your_data.gpkg')
# Values probably already exist per AGEB or manzana:
# table = pd.read_csv('../data/raw/your_file.csv',
#                     dtype={'CVEGEO': str})
# native = table.rename(columns={'CVEGEO': 'geo_id',
#                                'your_column': 'value'}
#                       )[['geo_id', 'value']]

native_112 = None  # <- assign your result here

In [ ]:
# 3. Harmonise, validate, deliver ------------------------------
harmonised_112 = uli.harmonise(
    native_112,
    native_scale=NATIVE_SCALE,
    method=METHOD,
)
results_112 = uli.label(
    harmonised_112,
    meta_112,
    measure_id='jobs__quantity',
)
# Several measures?  Label each and combine:
#   results = uli.assemble([results_a, results_b])

print(uli.check(results_112, meta_112))
# uli.write_indicator(results_112, meta_112)

---
## Check what this work package has delivered

In [ ]:
delivered, catalogue = uli.collect()
if len(catalogue):
    display(catalogue)
    print(delivered.groupby(['indicator_code', 'geo_level']).size())
else:
    print('Nothing delivered yet.')

In [ ]:
# Sanity-check a delivered measure on a map before you call it done.
# MEASURE = 'your_indicator_code__quantity'
# LEVEL = 'manzana'
# units = uli.geography.load(LEVEL).merge(
#     delivered.query('measure_id == @MEASURE and geo_level == @LEVEL'),
#     on='geo_id', how='left')
# ax = units.plot(column='value', legend=True, figsize=(11, 8),
#                 missing_kwds={'color': 'lightgrey'})
# condesa = uli.geography.load('condesa_fraccionamiento')
# condesa.boundary.plot(ax=ax, color='red', linewidth=1)
# ax.set_title(MEASURE)
# ax.set_axis_off()